# NB01 - Real images + preprocess

**Run once, after NB00.** Streams ~25,000 images from COCO and ~25,000 from ImageNet, pushes each through the canonical preprocess, and writes them to `real/*.parquet` with `real_NNNNNN` ids. **No generation happens here.**

### Prerequisites
- NB00 has been run (repo + `config.json` + library exist).
- **Internet: ON.**
- **GPU: not needed** - this stage is CPU/IO bound, so turn the GPU **off** to save quota.
- ImageNet terms accepted on your token's account.

Resume-safe: re-running continues from where it stopped (it counts what already exists). It ends with a **verifier** that checks everything is correct.

In [8]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
pip("huggingface_hub>=0.23", "datasets", "pyarrow", "pillow")
print("deps installed")

deps installed


In [9]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"   # <--- SAME as NB00

import os
from huggingface_hub import HfApi, hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass
    return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()

# download + import the shared library (the SAME code NB00 uploaded)
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
print("library + config loaded; pipeline", C.PIPELINE_VERSION, "| targets:", cfg["real_sources"])

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library + config loaded; pipeline 1.1 | targets: {'coco': 25000, 'imagenet': 25000}


## Acquire + preprocess
We stream each source (no full download), center-crop, resize to 512, JPEG-equalise and store as PNG via `canonical_preprocess`. Each image gets a unique `real_NNNNNN` id; `source_real_id == image_id` for real rows. Exact duplicate bytes (if any slip in) are removed later in NB09.

In [10]:
from datasets import load_dataset

api = C.HfApi(token=TOKEN)
targets = cfg["real_sources"]
SOURCE_HF = cfg["real_source_ids"]

# --- resume: count what already exists, find the max id ---
existing = {s: 0 for s in targets}
max_idx = 0
for f in C.list_shards(REPO_ID, "real/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_dataset", "image_id"])
    for s in t.column("source_dataset").to_pylist():
        if s in existing: existing[s] += 1
    for iid in t.column("image_id").to_pylist():
        try: max_idx = max(max_idx, int(str(iid).split("_")[-1]))
        except Exception: pass
gid = max_idx
print("existing per source:", existing, "| max id:", max_idx)

def iter_source(name):
    hfid = SOURCE_HF[name]
    if name == "coco":
        ds = load_dataset(hfid, split="train", streaming=True, trust_remote_code=True)
    else:  # imagenet (gated; needs token + accepted terms)
        ds = load_dataset(hfid, split="train", streaming=True, token=TOKEN)
    for ex in ds:
        img = ex.get("image")
        if img is not None:
            yield img

writer = C.ShardWriter(api, REPO_ID, "real/",
                       commit_interval=cfg["commit_interval_seconds"],
                       max_rows=cfg["batch_size"], token=TOKEN)
try:
    for name, want in targets.items():
        have = existing[name]
        if have >= want:
            print(f"{name}: already complete ({have}/{want})"); continue
        need = want - have
        print(f"{name}: producing {need} more ...")
        src = iter_source(name)
        for _ in range(have):                 # skip already-consumed (best effort)
            try: next(src)
            except StopIteration: break
        produced = 0
        for img in src:
            if produced >= need: break
            try:
                ow, oh = img.size
                png = C.canonical_preprocess(img)
            except Exception:
                continue                       # skip unreadable image
            gid += 1
            iid = f"real_{gid:06d}"
            row = C.empty_row()
            row.update(dict(image_id=iid, source_real_id=iid, label=0, generator="real",
                            source_dataset=name, prompt=None, image=png,
                            width=C.TARGET, height=C.TARGET,
                            orig_width=int(ow or 0), orig_height=int(oh or 0),
                            pipeline_version=C.PIPELINE_VERSION,
                            sha256=C.sha256_bytes(png), created_utc=C.now_utc()))
            writer.add(row); produced += 1
            if produced % 500 == 0:
                print(f"  {name}: {have + produced}/{want}")
            writer.maybe_flush()
        print(f"{name}: produced {produced} this run")
finally:
    writer.close()
print("real stage upload complete.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HuggingFaceM4/COCO' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


existing per source: {'coco': 0, 'imagenet': 0} | max id: 0
coco: producing 25000 more ...


README.md: 0.00B [00:00, ?B/s]

COCO.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found COCO.py

## Verifier
This checks the whole real stage: row counts vs targets, label/generator correctness, id uniqueness, the `source_real_id == image_id` invariant, and - by decoding a real sample - that every image is a 512x512 RGB PNG with no EXIF/ICC and a matching sha256. Read the **RESULT** line.

In [ ]:
ok = C.verify_real_stage(REPO_ID, TOKEN, cfg)
print('\nverifier returned:', ok)